In [ ]:
# API-key setup — DO NOT hard-code your key in this cell.

import os


from dotenv import load_dotenv

load_dotenv()
API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
# from google.colab import userdata
# API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",  # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"  # or your provider's model name

print("Client ready.")

Client ready.


In [2]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(
    user_prompt,
    system_prompt="You are a helpful assistant.",
    temperature=0.7,
    max_tokens=500,
):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

In [ ]:
raw_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What month are we in?"},
    ],
    temperature=0.7,
    max_tokens=500,
)
print(raw_response.choices[0].message.content)
print(raw_response.usage)

We are currently in August.
CompletionUsage(completion_tokens=7, prompt_tokens=47, total_tokens=54, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.051505259, prompt_time=0.004161485, completion_time=0.024497392, total_time=0.028658877)


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*

The system role sets the behaviour or role of the model for the conversation. The system role tells the model how to act.The user role is the user request or input for the model to act on.\

{"role": "system", "content": "You are an expert in geography."},
{"role": "user", "content": "What is the capital of Panama?"},


*2. What is a token, roughly? Why do API providers bill per token rather than per request?*
A token is a small chunk of text that a model reads or writes. API providers bill per token rather than per request because one request might be 5 words long while another may be a 50 page textbook. A per request fee would undercharge long tasks and overcharge short tasks.

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

In [ ]:
question = "Suggest a name for a savings product for market traders in Accra."
for i in range(5):
    answer = ask_llm(question, temperature=0.0)
    print(f"{i + 1}. {answer}\n")

for i in range(5):
    answer = ask_llm(question, temperature=1.2)
    print(f"{i + 1}. {answer}\n")

1. Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth, which could appeal to market traders.
3. **Sika Souce**: "Sika" means "money" in the Akan language, which is widely spoken in Ghana. "Souce" is a play on the word "source," implying a reliable and trustworthy savings product.
4. **Market Mobi**: This name incorporates "mobi," short for mobile, which could suggest a convenient and accessible savings product for market traders who are always on the go.
5. **Kokroko Savings**: "Kokroko" is a Ghanaian word that means "honest" or "trustworthy." This name could convey a sense of reliability and security, which is important for a savings product.
6. **Adanfo Account**: "Adanfo" means "friends" or "partners" in the Akan language. This name could suggest a savings p

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

At both temperatures = 0.0, and 1.2, there was noticeable variation between calls, however several words/ names were repeated throughout both tests. ("Makola","Trader's","Sika"). For the loan decision support system, temperature = 0.0 would be most appropriate because T<1 minimizes randomness and variation, and for a decision support system, we would need certain answers and responses.

In [7]:
LETTERS = {
    "L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",
    "L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",
    "L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",
    "L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",
    "L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",
    "L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20,
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15,
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12,
    },
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

In [ ]:
SUMMARY_PROMPT_V1 = "Summarize this:"


def summarize_v1(letter_text):
    return ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}", temperature=0.0)


print("-----V1 — L002 ----")
print(summarize_v1(LETTERS["L002"]))
print()
print("----V1 — L006 -----")
print(summarize_v1(LETTERS["L006"]))
print()

-----V1 — L002 ----
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when his finances recover.

----V1 — L006 -----
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.



In [ ]:
SUMMARY_SYSTEM_PROMPT_V2 = """You are an assistant to a microfinance loan officer.
Your job is to summarize loan application letters accurately and neutrally.
Rules:
- Only use information explicitly stated in the letter, never invent details.
- Be factual and neutral in tone.
- Keep the summary to 3-4 sentences.
- If key information (amount, purpose, collateral) is missing from the
  letter, do not fill it in, simply omit it or note that it wasn't provided."""


def summarize_v2(letter_text):
    user_prompt = f"Summarize this loan application:\n\n{letter_text}"
    return ask_llm(
        user_prompt,
        system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
        temperature=0.0,
    )


print("-----V2 — L002 -----")
print(summarize_v2(LETTERS["L002"]))
print()

print("-----V2 — L006 ----")
print(summarize_v2(LETTERS["L006"]))
print()

-----V2 — L002 -----
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. The purpose of the loan is to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to improve after the festive season. No collateral is currently available to secure the loan, and a specific repayment plan is not provided.

-----V2 — L006 ----
Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business. He is 22 years old and claims to be business-minded, although he has not yet started any of these ventures. Kofi intends to repay the loan within one year, expecting his businesses to be successful by then. He does not offer any collateral, instead assuring that he is trustworthy.



In [15]:
print("----Comparison----")
for letter_id in ["L002", "L006"]:
    print(f"LETTER {letter_id}")
    print("-------------------------------------------")
    print(f"\n--- V1 (naive) ---")
    print(summarize_v1(LETTERS[letter_id]))
    print(f"\n--- V2 (proper) ---")
    print(summarize_v2(LETTERS[letter_id]))
    print()

----Comparison----
LETTER L002
-------------------------------------------

--- V1 (naive) ---
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when possible, despite not having collateral.

--- V2 (proper) ---
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. The purpose of the loan is to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season, which he believes will enable him to repay the loan, although he did not specify a repayment schedule. Mr. Boateng does not have collateral to offer at the moment.

LETTER L006
-------------------------------------------

--- V1 (naive) ---
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a prov

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*

On L002, Kwame mentioned, "I can pay back whenever the money comes. I do not have collateral at the moment but God willing everything will be fine.". V1 wrote that Kwame is "willing to repay the oan when possible"while V2 states plainly that "he did not specify a repayment schedule". Here, V2 accurately summarizes the neutral meaning behind Kwame's sentence while V1 just paraphrases the sentence.

*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

"No invented details" is an essential instruction in this application to prevent hallucination in the model. Hallucination is where the model generates confident, but false or ungrounded content. A loan decision-support system should base its responses only on information provided by the applicant, if the llm assumes facts, it makes the loan officer seem unreliable. 

In [ ]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

In [ ]:
import json
import pandas as pd

EXTRACT_PROMPT = """You are a data extraction assistant for a microfinance loan officer.
You will be given a loan application letter. Extract information into a JSON object with
EXACTLY these six keys, and nothing else:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

Rules:
- If a field is not explicitly stated in the letter, use null. Do not guess or infer.
- Output ONLY the JSON object. No explanation, no markdown code fences, no extra text.
- has_collateral_or_guarantor should be true only if the letter explicitly mentions
  collateral, a guarantor, or a pledged asset.

Example:

Letter:
"Dear Sir/Madam, my name is John Aheto. I have a restaurant in Tema and would like GHS 10,000 to
buy a new gas cooker and expand seating in the restaurant. My profit is about GHS 1,200 a month. I can repay
GHS 700 monthly for 15 months. My brother is a police officer, he will stand as my guarantor."

Output:
{
    "applicant_name": "John Aheto", 
    "amount_ghs": 10000, 
    "purpose": "buy new gas cooker and expand seating", 
    "monthly_profit_ghs": 1200, 
    "has_collateral_or_guarantor": true, 
    "repayment_months": 15
}

Now extract the fields from this loan application:

LETTER_TEXT_HERE
"""


def extract_fields(letter_text):
    prompt = EXTRACT_PROMPT.replace("LETTER_TEXT_HERE", letter_text)

    result = None

    try:
        result = ask_llm(prompt, temperature=0.0).strip()

        if result.startswith("```json"):
            result = result[7:].strip()
        elif result.startswith("```"):
            result = result[3:].strip()
        if result.endswith("```"):
            result = result[:-3].strip()
        data = json.loads(result)

        if not isinstance(data, dict):
            print("WARNING: Response is not a JSON object.")
            return None

        required_keys = {
            "applicant_name",
            "amount_ghs",
            "purpose",
            "monthly_profit_ghs",
            "has_collateral_or_guarantor",
            "repayment_months",
        }

        if set(data.keys()) != required_keys:
            print("WARNING: JSON does not contain exactly the required keys.")
            print("Returned keys:", list(data.keys()))
            return None

        return data

    except json.JSONDecodeError:
        print("WARNING: Could not parse model response as JSON.")
        print("Model response:")
        print(result)
        return None

    except Exception as e:
        print(f"WARNING: Extraction failed: {e}")
        return None


print("Testing extraction on L001...\n")

test_result = extract_fields(LETTERS["L001"])

print("Extracted result:")
print(test_result)

results = []

print("\n" + "=" * 70)
print("EXTRACTING ALL SIX LETTERS")
print("=" * 70)

for letter_id, letter_text in LETTERS.items():
    print(f"\nExtracting {letter_id}...")

    extracted = extract_fields(letter_text)

    if extracted is not None:
        extracted["letter_id"] = letter_id
        results.append(extracted)
        print("Extraction successful.")
    else:
        print("Extraction failed.")


df_extracted = pd.DataFrame(results)

columns = [
    "letter_id",
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]

if not df_extracted.empty:
    df_extracted = df_extracted[columns]

print("\n" + "=" * 70)
print("FINAL EXTRACTED DATA")
print("=" * 70)

display(df_extracted)

Testing extraction on L001...

Extracted result:
{'applicant_name': 'Akosua Mensah', 'amount_ghs': 8000, 'purpose': 'buy a deep freezer and expand into frozen foods', 'monthly_profit_ghs': 900, 'has_collateral_or_guarantor': True, 'repayment_months': 20}

EXTRACTING ALL SIX LETTERS

Extracting L001...
Extraction successful.

Extracting L002...
Extraction successful.

Extracting L003...
Extraction successful.

Extracting L004...
Extraction successful.

Extracting L005...
Extraction successful.

Extracting L006...
Extraction successful.

FINAL EXTRACTED DATA


,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,for feed and 500 new layers,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*

The few-shot example should not come from the six letters because it could leak expected answers. If answers get leaked to the model, it stops learning and simply copies the same values from that letter. 

*2. Why "use null, do not guess" — what did the model do without that instruction?*

Without that instruction, the model may try to fill in missing information based on assumptions. For instance, from the six letters, L002 did not state Kwame's monthly profit. Without the instruction to use null, the model would have tried to assume vlaues.

*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*
Temperature>1 makes the model precise and factual while temperature >1 makes the model more random/ creative, and for extraction, we do not need any randomness or creativity, we nee the model to extract factual values and information from the letters.


In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

In [ ]:
BRIEF_PROMPT = """
You are an assistant supporting a microfinance loan officer. 
You help officers quickly review loan applications by producing a structured brief.

You will be given:
1. The original loan application.
2. The extracted JSON information.

RULES:
- Every point must be grounded in the letter or the extracted JSON. Do not invent facts
  not present in either.
- Under "Suggested Next Step", choose ONE concrete next action such as
  "invite for interview", "request supporting documents", or "flag for senior review".
- You must NEVER output a final lending decision such as "approve" or "reject" the loan.
  Final approval or rejection decisions are made only by human loan officers, not by you.
  Your job is to support their review, not replace it.
- Do not invent, assume, or infer information. If information is missing, explicitly identify it.
- Every strength and risk must be directly supported by the letter.
- Do not turn opinions or claims made by the applicant into verified facts.
- Do not treat age, enthusiasm, optimism, trustworthiness, or similar
  personal claims as evidence of repayment ability unless independently
  supported by information in the letter.
- Do not describe something as a strength merely because it exists.
- If a fact could reasonably be either positive or negative, describe it
  neutrally rather than labeling it a strength.
- Do not use information that is not in the application.


Use EXACTLY these four sections:

## 1. Strengths
- List only concrete, evidence-based strengths.
- Each strength must be directly supported by the application.
- Do not infer future success from age, enthusiasm, optimism, or intentions.

## 2. Risks / Red Flags
- List concrete risks or concerns directly supported by the application.
- Focus on financial uncertainty, repayment concerns, lack of business history,
  lack of collateral, existing debts, or other relevant evidence.
- Do not exaggerate or invent risks.

## 3. Missing Information
- List information or documents that the loan officer should request
  before making a decision.
- Examples include business records, bank statements, proof of income,
  business registration, collateral documentation, guarantor information,
  or a business plan when relevant.
- Only request information that is relevant to the application.

## 4. Suggested Next Step
- Suggest ONE practical action for the human loan officer.
- Examples:
  "Invite the applicant for an interview."
  "Request supporting financial documents."
  "Verify the guarantor and collateral."
  "Request a detailed business plan."
  "Flag for senior review."
- NEVER say "approve" or "reject".

ORIGINAL LOAN APPLICATION:
{letter}

EXTRACTED INFORMATION:
{extracted_json}
"""


def generate_brief(letter_text, extracted_data):
    extracted_json = json.dumps(extracted_data, indent=2)
    prompt = BRIEF_PROMPT.replace("{letter}", letter_text).replace(
        "{extracted_json}", extracted_json
    )

    try:
        response = ask_llm(prompt, temperature=0.0)
        return response.strip()
    except Exception as e:
        print(f"WARNING: Could not generate brief: {e}")
        return None


briefs = {}

for letter_id, letter_text in LETTERS.items():
    print(f"Generating brief for {letter_id}...")
    row = df_extracted[df_extracted["letter_id"] == letter_id]
    if row.empty:
        print(f"WARNING: No extracted data found for {letter_id}")
        briefs[letter_id] = None
        continue
    extracted_data = row.iloc[0].to_dict()

    extracted_data.pop("letter_id", None)
    briefs[letter_id] = generate_brief(letter_text, extracted_data)


for letter_id in ["L001", "L002", "L006"]:
    print(f"LOAN BRIEF — {letter_id}")
    print("-------------------------------------------")

    if briefs[letter_id]:
        print(briefs[letter_id])
    else:
        print("No brief generated.")

Generating brief for L001...
Generating brief for L002...
Generating brief for L003...
Generating brief for L004...
Generating brief for L005...
Generating brief for L006...
LOAN BRIEF — L001
-------------------------------------------
## 1. Strengths
- The applicant has a established business history, having sold provisions at Makola Market for 12 years.
- The applicant has a consistent income, with a monthly profit of GHS 900 from their current stall.
- The applicant has a savings history, having saved GHS 2,500 with the susu scheme over two years without missing a contribution.
- The applicant has a guarantor, their sister, who is a teacher.

## 2. Risks / Red Flags
- The applicant is expanding into a new area of business (frozen foods) which may come with unknown challenges and risks.
- The repayment plan is based on the applicant's ability to maintain their current monthly profit and potentially increase it to cover the loan repayments of GHS 450 per month.

## 3. Missing Informat

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the system identify the right strengths and red flags in each?*

For L003's strengths, Efua Darko has a registered dressmaking business, employs three apprentices, has an 18-month sales record, she earns an average monthly profit of GHS 2,800, and has a GHS 5,000 fixed deposit. These are concrete pieces of evidence that support the application. The main risk should be that the requested GHS 15,000 is being used for expansion and that the officer still needs to verify the sales records, business performance, and fixed deposit. For L006's records, Kofi has no collateral or guarantor, he has not started any of the proposed businesses yet, and has not provided a concrete repayment strategy. The system should request for a detailed business plan for each of the proposed businesses, projected financials, and strategy for repayment.

*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and one ethical reason.*

We forbid the model from outputting "approve/reject" because the model cannot verify anythign from the documents. it has no way to prove that fixed deposits stated by the customer actually exist. An ethical reason is that lending decisions are important and they affect people's livelihoods, forbidding the model from outputting approve/rejectt keeps the human accountable for the outcome.

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

In [ ]:
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]

gold_letter_ids = ["L001", "L003", "L006"]


def values_match(field, extracted_val, gold_val):
    if gold_val is None and (extracted_val is None or pd.isna(extracted_val)):
        return True
    if gold_val is None or extracted_val is None or pd.isna(extracted_val):
        return False

    if field == "applicant_name":
        return str(extracted_val).strip().lower() == str(gold_val).strip().lower()

    if field == "purpose":
        return str(extracted_val).strip().lower() == str(gold_val).strip().lower()

    if field == "has_collateral_or_guarantor":
        return bool(extracted_val) == bool(gold_val)

    try:
        return float(extracted_val) == float(gold_val)
    except (ValueError, TypeError):
        return False


rows = []
for field in fields:
    row = {"field": field}
    correct_count = 0

    for letter_id in gold_letter_ids:
        gold_val = GOLD[letter_id][field]
        extracted_row = df_extracted[df_extracted["letter_id"] == letter_id]

        if extracted_row.empty:
            extracted_val = None
        else:
            extracted_val = extracted_row.iloc[0][field]

        is_correct = values_match(field, extracted_val, gold_val)
        correct_count += int(is_correct)

        # Show what was extracted, with a checkmark/cross for correctness
        mark = "✓" if is_correct else "✗"
        row[letter_id] = f"{extracted_val} {mark}"

    row["accuracy"] = f"{correct_count}/3 ({correct_count / 3:.0%})"
    rows.append(row)

comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df[["field", "L001", "L003", "L006", "accuracy"]]

display(comparison_df)

,field,L001,L003,L006,accuracy
0,applicant_name,Akosua Mensah ✓,Efua Darko ✓,Kofi ✓,3/3 (100%)
1,amount_ghs,8000 ✓,15000 ✓,50000 ✓,3/3 (100%)
2,purpose,buy a deep freezer and expand into frozen foods ✗,purchase two industrial sewing machines and fa...,"start a car washing business, a provision shop...",0/3 (0%)
3,monthly_profit_ghs,900.0 ✓,2800.0 ✓,nan ✓,3/3 (100%)
4,has_collateral_or_guarantor,True ✓,True ✓,False ✓,3/3 (100%)
5,repayment_months,20.0 ✓,15.0 ✓,12.0 ✓,3/3 (100%)


In [ ]:
import json


def extract_fields_temp(letter_text, temperature=0.0):
    prompt = EXTRACT_PROMPT.replace("LETTER_TEXT_HERE", letter_text)
    result = None
    try:
        result = ask_llm(prompt, temperature=temperature).strip()

        if result.startswith("```json"):
            result = result[7:].strip()
        elif result.startswith("```"):
            result = result[3:].strip()
        if result.endswith("```"):
            result = result[:-3].strip()

        data = json.loads(result)

        if not isinstance(data, dict):
            return None

        required_keys = {
            "applicant_name",
            "amount_ghs",
            "purpose",
            "monthly_profit_ghs",
            "has_collateral_or_guarantor",
            "repayment_months",
        }
        if set(data.keys()) != required_keys:
            return None

        return data

    except Exception:
        return None


letter_text = LETTERS["L004"]

runs_temp0 = []
runs_temp1 = []

print("Running 5x at temperature=0.0...")
for i in range(5):
    result = extract_fields_temp(letter_text, temperature=0.0)
    runs_temp0.append(result)
    print(f"  Run {i + 1}: {'valid JSON' if result is not None else 'PARSE FAILED'}")

print("\nRunning 5x at temperature=1.0...")
for i in range(5):
    result = extract_fields_temp(letter_text, temperature=1.0)
    runs_temp1.append(result)
    print(f"  Run {i + 1}: {'valid JSON' if result is not None else 'PARSE FAILED'}")


def summarize_runs(runs, label):
    valid_runs = [r for r in runs if r is not None]
    n_valid = len(valid_runs)

    canonical_strings = [json.dumps(r, sort_keys=True) for r in valid_runs]
    unique_strings = set(canonical_strings)

    print(f"\n=== {label} ===")
    print(f"Valid JSON: {n_valid}/5")
    print(f"Unique result variants among valid runs: {len(unique_strings)}")
    if n_valid > 0:
        from collections import Counter

        counts = Counter(canonical_strings)
        most_common_str, most_common_count = counts.most_common(1)[0]
        print(f"Most common result appeared {most_common_count}/{n_valid} times")
        if len(unique_strings) == 1:
            print("→ All valid runs were IDENTICAL.")
        else:
            print("→ Valid runs DIFFERED across runs.")
    return {
        "n_valid": n_valid,
        "n_unique": len(unique_strings),
    }


print("\n" + "=" * 70)
print("CONSISTENCY REPORT — L004")
print("=" * 70)

summary_temp0 = summarize_runs(runs_temp0, "Temperature = 0.0")
summary_temp1 = summarize_runs(runs_temp1, "Temperature = 1.0")

Running 5x at temperature=0.0...
  Run 1: valid JSON
  Run 2: valid JSON
  Run 3: valid JSON
  Run 4: valid JSON
  Run 5: valid JSON

Running 5x at temperature=1.0...
  Run 1: valid JSON
  Run 2: valid JSON
  Run 3: valid JSON
  Run 4: valid JSON
  Run 5: valid JSON

CONSISTENCY REPORT — L004

=== Temperature = 0.0 ===
Valid JSON: 5/5
Unique result variants among valid runs: 1
Most common result appeared 5/5 times
→ All valid runs were IDENTICAL.

=== Temperature = 1.0 ===
Valid JSON: 5/5
Unique result variants among valid runs: 1
Most common result appeared 5/5 times
→ All valid runs were IDENTICAL.


In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

In [ ]:
adversarial_prompt_1 = (
    "What is the applicant's credit score? Answer based only on the letter below.\n\n"
    f"{LETTERS['L001']}"
)

test1_output = ask_llm(
    adversarial_prompt_1,
    system_prompt=SUMMARY_SYSTEM_PROMPT_V2,
    temperature=0.0,
)

print("=== ADVERSARIAL TEST 1 — asking about credit score (not in letter) ===")
print(test1_output)

=== ADVERSARIAL TEST 1 — asking about credit score (not in letter) ===
The letter does not mention the applicant's credit score. It provides information about the applicant's business, savings, and proposed loan repayment plan, but does not include a credit score. The applicant mentions that she has never missed a contribution to the susu scheme, which suggests a positive savings history, but a credit score is not provided. The applicant's financial history is described in terms of her savings and proposed repayment plan.


In [23]:
weather_report = """Accra Weather Update — Tuesday
Expect partly cloudy skies today with a high of 31°C and a low of 24°C.
Humidity will remain high at around 80%, with a slight chance of afternoon
showers along the coast. Winds light, from the southwest at 10-15 km/h.
UV index: high. Tomorrow's forecast: mostly sunny with similar temperatures."""

test2_output_raw = extract_fields_temp(weather_report, temperature=0.0)

print("=== ADVERSARIAL TEST 2 — feeding extractor a weather report ===")
print(test2_output_raw)

=== ADVERSARIAL TEST 2 — feeding extractor a weather report ===
{'applicant_name': None, 'amount_ghs': None, 'purpose': None, 'monthly_profit_ghs': None, 'has_collateral_or_guarantor': None, 'repayment_months': None}


Test 1: model explicitly states the credit score is not mentioned in the letter. Final verdict is pass.

Test 2: model inputs "None" in all fields due to missing required content. Final verdict is pass.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*

The purpose field was the hardest with an accuracy of 0%. However, this is not a limitation of the extraction system. Purpose is free text with no single correct phrasing, but the evaluation used exact case insensitive string matching.

*2. What did the reliability experiment show about temperature and production systems?*

At temperature 0.0, all 5/5 runs produced valid JSON and all five outputs were identical, giving 1 unique output. However, at temperature 1.0, all 5/5 runs still produced valid JSON, but three distinct output variants appeared meaning the results varied between runs.

*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system design around it) reduce the risk?*

No, both adversarial tests passed and there was no hallucination. The instructions “use null, do not guess” and “do not invent information” helped reduce hallucination.


**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions with your system, who could be unfairly harmed, and how? Consider applicants who write poorly in English but run solid businesses.*

Yes, if the bank fully automated decisions with the system, applicants who write poorly in English but run solid businesses may be unfairly harmed. The fullt automated system may judge applicants based on how well the write english, so a trader who runs a legit, profitable, and reliable business, but writes a bad letter may be rejected because the model may miss important information and deny the applicant, while a human reviewer could look past the bad english.  

*2. Loan letters contain personal data. What are the implications of sending them to a third-party API in another country? What would you check before deploying this at a real Ghanaian microfinance institution?*

These loan letters contain personal and importtant financial information (names, income, guarantors) and sending that to a foreign country means it leaves Ghana's legal jurisdiction and becomes subject to that country's laws instead, creating privacy, security, and legal risks. Before deploying,I would check Ghana's Data Protection Act requirements, and heck the API provider's data retention policy.

*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

The first safeguard would be mandatory human review before any decision is communicated. Every extraction, brief, and officer decision should also be logged with timestamps to allow reviews if applicants believe they were wrongly judged.

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?

In lab 3, we tuned learning rate, dropout, and capcity against a validation metric, and here we tuned system prompts, constraints, and a few-shot examples against accuracy and hallucinttion checks. The differenece is that, yperparameters are numeric and have a predictable effect on a fixed model, while prompts are natural language, so small wording changes can cause large shifts in model behaviour.


2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?

No. I would not trust the system to run unattended, it is intended to produce a brief for the human to give the final decision so the human beign can be held acounttable. The model cannot verify anything from the documents. it has no way to prove that fixed deposits stated by the customer actually exist. 


3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
From my response.usage, a simple test call used 54 tokens total, meaning 54,000 tokens per month, but a real application would be around 1000 tokens once you account for both the extraction step and the brief-generation step. Scaling that to 1000 applications a month gives roughly 1000000 tokens per month.


   
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?


Calling an API beats training your own model here because a foundation model already has broad language understanding and world knowledge built in you get strong performance from a few hours of prompt engineering. However, training your own model could be preferable if the institution required specialized terminology, or local languages needed to complete control over the model and data.

